In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Chargement des données
df = pd.read_excel('Consumer_Complaints.xlsx')
print(f"Dimensions initiales : {df.shape}")

# 2. Nettoyage et conversion des dates
df['Date submitted'] = pd.to_datetime(df['Date submitted'])
df['Date received'] = pd.to_datetime(df['Date received'])

# 3. Feature Engineering : Calcul du délai de transmission (en jours)
df['Processing_Delay'] = (df['Date received'] - df['Date submitted']).dt.days

# Extraction de l'année et du mois pour l'EDA temporelle
df['Year'] = df['Date submitted'].dt.year
df['YearMonth'] = df['Date submitted'].dt.to_period('M')

# Aperçu rapide des valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())

Dimensions initiales : (62516, 12)

Valeurs manquantes par colonne :
Complaint ID                        0
Submitted via                       0
Date submitted                      0
Date received                       0
State                               0
Product                             0
Sub-product                         7
Issue                               0
Sub-issue                       10858
Company public response          2175
Company response to consumer        0
Timely response?                 1494
Processing_Delay                    0
Year                                0
YearMonth                           0
dtype: int64


In [5]:
# 3. CRÉATION DES VISUALISATIONS (EDA)
# ==========================================
os.makedirs('figures', exist_ok=True)

# Graphique 1 : Évolution annuelle du volume des réclamations
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Year', hue='Year', palette='viridis', legend=False)
plt.title('Volume des réclamations par an (2017-2023)')
plt.xlabel('Année')
plt.ylabel('Nombre de réclamations')
plt.savefig('figures/01_complaints_by_year.png', bbox_inches='tight')
plt.close()

# Graphique 2 : Top 5 des produits financiers concernés
plt.figure(figsize=(10, 5))
top_products = df['Product'].value_counts().index[:5]
sns.countplot(data=df[df['Product'].isin(top_products)], y='Product', order=top_products, hue='Product', palette='magma', legend=False)
plt.title('Top 5 des produits financiers concernés')
plt.xlabel('Nombre de réclamations')
plt.ylabel('Produit')
plt.savefig('figures/02_top_products.png', bbox_inches='tight')
plt.close()

# Graphique 3 : Distribution des réponses de la banque aux consommateurs
plt.figure(figsize=(10, 5))
sns.countplot(data=df, y='Company response to consumer', order=df['Company response to consumer'].value_counts().index, hue='Company response to consumer', palette='crest', legend=False)
plt.title('Distribution des réponses de la banque aux consommateurs')
plt.xlabel('Nombre de réclamations')
plt.ylabel('Type de réponse')
plt.savefig('figures/03_company_responses.png', bbox_inches='tight')
plt.close()

# Graphique 4 : Distribution des délais de transmission (Processing Delay)
plt.figure(figsize=(10, 5))
sns.histplot(df['Processing_Delay'], bins=30, kde=True, color='teal')
plt.title('Distribution des délais de transmission du CFPB (en jours)')
plt.xlabel('Délai (jours)')
plt.ylabel('Fréquence')
plt.savefig('figures/04_processing_delay_dist.png', bbox_inches='tight')
plt.close()

# Graphique 5 : Répartition des canaux de soumission
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Submitted via', order=df['Submitted via'].value_counts().index, hue='Submitted via', palette='Spectral', legend=False)
plt.title('Répartition des canaux de soumission des réclamations')
plt.xlabel('Canal de soumission')
plt.ylabel('Nombre de réclamations')
plt.xticks(rotation=45)
plt.savefig('figures/05_submitted_via.png', bbox_inches='tight')
plt.close()

# Graphique 6 : Top 10 des États américains les plus concernés
plt.figure(figsize=(12, 6))
top_states = df['State'].value_counts().index[:10]
sns.countplot(data=df[df['State'].isin(top_states)], y='State', order=top_states, hue='State', palette='coolwarm', legend=False)
plt.title('Top 10 des États américains les plus concernés')
plt.xlabel('Nombre de réclamations')
plt.ylabel('État')
plt.savefig('figures/06_top_states.png', bbox_inches='tight')
plt.close()

print("\nTous les graphiques ont été générés et sauvegardés avec succès dans le dossier 'figures/'.")


Tous les graphiques ont été générés et sauvegardés avec succès dans le dossier 'figures/'.


In [3]:
from scipy.stats import chi2_contingency

# Nettoyage rapide pour ne garder que les lignes sans valeurs manquantes sur ces colonnes
df_clean = df.dropna(subset=['Product', 'Timely response?'])

# Tableau croisé
contingency_table = pd.crosstab(df_clean['Product'], df_clean['Timely response?'])

# Application du test du Chi-2
chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"Statistique du Chi-2 : {chi2:.2f}")
print(f"P-value : {p}")

Statistique du Chi-2 : 414.26
P-value : 1.6617373943120047e-84
